In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session



In [ ]:
# Load base datasets
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv', parse_dates=['date'])
test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv', parse_dates=['date'])

stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv', parse_dates=['date'])
oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv', parse_dates=['date'])
holidays = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv', parse_dates=['date'])

# Fix oil missing values (future-proof)
oil['dcoilwtico'] = oil['dcoilwtico'].ffill()

# Merge store metadata
train = train.merge(stores, on='store_nbr', how='left')
test = test.merge(stores, on='store_nbr', how='left')

# Merge transactions
train = train.merge(transactions, on=['date', 'store_nbr'], how='left')
test = test.merge(transactions, on=['date', 'store_nbr'], how='left')

# Merge oil
train = train.merge(oil, on='date', how='left')
test = test.merge(oil, on='date', how='left')

# Preprocess holidays: keep only non-transferred & make single entry per date
holidays = holidays[holidays['transferred'] == False]
holidays = holidays.groupby('date').first().reset_index()

# Explicit suffixes to avoid 'type_x'/'type_y' issues
train = train.merge(
    holidays[['date', 'type', 'locale', 'locale_name', 'description']],
    on='date', how='left', suffixes=('', '_holiday')
)
test = test.merge(
    holidays[['date', 'type', 'locale', 'locale_name', 'description']],
    on='date', how='left', suffixes=('', '_holiday')
)

print("Train shape:", train.shape)
print("Test shape:", test.shape)



In [ ]:
def preprocess(df):
    # Date features
    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['dayofweek'] = df['date'].dt.dayofweek
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    
    # Encode store type
    df['store_type'] = df['type'].map({'A': 0, 'B': 1, 'C': 2, 'D': 3})

    # Holiday indicator from merged holidays table
    df['holiday'] = df['type_holiday'].notnull().astype(int)

    # Fill missing values
    df['transactions'] = df['transactions'].fillna(0)
    df['dcoilwtico'] = df['dcoilwtico'].ffill()

    return df

# Apply preprocessing
train = preprocess(train)
test = preprocess(test)

# Drop noisy holiday columns
drop_cols = [col for col in ['type_holiday', 'locale_holiday', 'locale_name_holiday', 'description'] if col in train.columns]
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)


print("✅ Feature engineering done.")



In [ ]:
import lightgbm as lgb
from lightgbm import early_stopping
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np

# Target
TARGET = 'sales'

# Columns to drop
drop_cols = ['id', 'date', 'sales']
# Add this right below
lag_features = ['lag_1', 'lag_7', 'rolling_7', 'rolling_14', 'monthly_avg']
features = [col for col in train.columns if col not in drop_cols]
features += [f for f in lag_features if f not in features]

# Label encode object columns
for col in train.select_dtypes(include='object').columns:
    le = LabelEncoder()
    full_data = pd.concat([train[col], test[col]], axis=0)
    le.fit(full_data.fillna('NA'))
    train[col] = le.transform(train[col].fillna('NA'))
    test[col] = le.transform(test[col].fillna('NA'))

# Time-based split
cutoff_date = '2017-07-01'
train_set = train[train['date'] < cutoff_date]
val_set = train[train['date'] >= cutoff_date]

# LightGBM datasets
lgb_train = lgb.Dataset(train_set[features], label=train_set[TARGET])
lgb_val = lgb.Dataset(val_set[features], label=val_set[TARGET])

# LightGBM params
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'seed': 42
}

# Train with callback-based early stopping
# Import early stopping and logging
from lightgbm import early_stopping, log_evaluation

# Train LightGBM with callbacks only
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'valid'],
    callbacks=[
        early_stopping(stopping_rounds=50),
        log_evaluation(period=100)
    ]
)


# Predict on validation
val_pred = model.predict(val_set[features])
val_rmse = np.sqrt(mean_squared_error(val_set[TARGET], val_pred))
print(f"Validation RMSE: {val_rmse:.4f}")

# =====================
# ✅ Submission section
# =====================

# Predict on test
test_preds = model.predict(test[features])
test_preds = np.clip(test_preds, 0, None)  # no negative sales

# Prepare submission
sample_sub = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/sample_submission.csv')
sample_sub['sales'] = test_preds

# Save to Kaggle output path
output_path = '/kaggle/working/submission.csv'
sample_sub.to_csv(output_path, index=False)
print(f"✅ Submission saved to: {output_path}")

 


In [ ]:
# Make copies
train_lag = train.copy()
test_lag = test.copy()

# Combine for rolling features
full = pd.concat([train_lag, test_lag], sort=False).sort_values(['store_nbr', 'family', 'date'])

# Lag features
full['lag_1'] = full.groupby(['store_nbr', 'family'])['sales'].shift(1)
full['lag_7'] = full.groupby(['store_nbr', 'family'])['sales'].shift(7)

# Rolling means
full['rolling_7'] = full.groupby(['store_nbr', 'family'])['sales'].shift(1).rolling(7).mean()
full['rolling_14'] = full.groupby(['store_nbr', 'family'])['sales'].shift(1).rolling(14).mean()

# Optional: monthly average trend
full['monthly_avg'] = (
    full.groupby(['store_nbr', 'family', full['date'].dt.to_period("M")])['sales'].transform('mean')
)
full['monthly_avg'] = full['monthly_avg'].shift(1)

# Re-split
train = full[full['date'] < '2017-08-16'].copy()
test = full[full['date'] >= '2017-08-16'].copy()

# Drop NaNs introduced by rolling
train.dropna(inplace=True)

print("✅ Lag + rolling features added. Train shape:", train.shape, "| Test shape:", test.shape)
